In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import accuracy_score, log_loss
import xgboost as xgb

# Suppress warnings for cleaner output
import warnings
warnings.filterwarnings('ignore')

In [3]:
# Load the datasets
matches = pd.read_csv('matches.csv')
deliveries = pd.read_csv('deliveries.csv.zip')

# Calculate the total score for the 1st innings of every match
total_score_df = deliveries.groupby(['match_id', 'inning']).sum()['total_runs'].reset_index()
total_score_df = total_score_df[total_score_df['inning'] == 1]

# Merge the target score with the match details
match_df = matches.merge(total_score_df[['match_id', 'total_runs']], left_on='id', right_on='match_id')

# Keep only matches where the DLS method (rain delay) was NOT applied
match_df = match_df[match_df['dl_applied'] == 0]

In [4]:
# Standardize current IPL teams
teams = [
    'Sunrisers Hyderabad', 'Mumbai Indians', 'Royal Challengers Bangalore',
    'Kolkata Knight Riders', 'Punjab Kings', 'Chennai Super Kings',
    'Rajasthan Royals', 'Delhi Capitals', 'Gujarat Titans', 'Lucknow Super Giants'
]

# Replace old team names with current ones (e.g., Delhi Daredevils -> Delhi Capitals)
match_df['team1'] = match_df['team1'].str.replace('Delhi Daredevils', 'Delhi Capitals')
match_df['team2'] = match_df['team2'].str.replace('Delhi Daredevils', 'Delhi Capitals')
match_df['team1'] = match_df['team1'].str.replace('Deccan Chargers', 'Sunrisers Hyderabad')
match_df['team2'] = match_df['team2'].str.replace('Deccan Chargers', 'Sunrisers Hyderabad')
match_df['team1'] = match_df['team1'].str.replace('Kings XI Punjab', 'Punjab Kings')
match_df['team2'] = match_df['team2'].str.replace('Kings XI Punjab', 'Punjab Kings')

match_df = match_df[match_df['team1'].isin(teams)]
match_df = match_df[match_df['team2'].isin(teams)]

# Filter the deliveries data to only include the 2nd innings of our selected matches
match_df = match_df[['match_id', 'city', 'winner', 'total_runs']]
delivery_df = match_df.merge(deliveries, on='match_id')
delivery_df = delivery_df[delivery_df['inning'] == 2]

In [5]:
# 1. Current Score
delivery_df['current_score'] = delivery_df.groupby('match_id')['total_runs_y'].cumsum()

# 2. Runs Left (Target is 1st innings total + 1)
delivery_df['runs_left'] = (delivery_df['total_runs_x'] + 1) - delivery_df['current_score']

# 3. Balls Left
delivery_df['balls_left'] = 120 - (delivery_df['over'] * 6 + delivery_df['ball'])

# 4. Wickets Left
# Fill NaNs in player_dismissed with "0", else "1", then calculate cumulative sum
delivery_df['player_dismissed'] = delivery_df['player_dismissed'].fillna("0")
delivery_df['player_dismissed'] = delivery_df['player_dismissed'].apply(lambda x: x if x == "0" else "1")
delivery_df['player_dismissed'] = delivery_df['player_dismissed'].astype('int')

wickets_fallen = delivery_df.groupby('match_id')['player_dismissed'].cumsum()
delivery_df['wickets_left'] = 10 - wickets_fallen

# 5. Current Run Rate (CRR)
delivery_df['crr'] = (delivery_df['current_score'] * 6) / (120 - delivery_df['balls_left'])

# 6. Required Run Rate (RRR)
delivery_df['rrr'] = (delivery_df['runs_left'] * 6) / delivery_df['balls_left']

# 7. Target Variable (Result: 1 if batting team wins, 0 otherwise)
def result(row):
    return 1 if row['batting_team'] == row['winner'] else 0

delivery_df['result'] = delivery_df.apply(result, axis=1)

In [6]:
# Extract only the necessary features
final_df = delivery_df[['batting_team', 'bowling_team', 'city', 'runs_left', 'balls_left', 'wickets_left', 'total_runs_x', 'crr', 'rrr', 'result']]

# Drop null values (usually caused by missing 'city' data) and infinite values in RRR (when balls_left == 0)
final_df = final_df.dropna()
final_df = final_df[final_df['balls_left'] != 0]

# Split the data
X = final_df.drop('result', axis=1)
y = final_df['result']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [7]:
# Preprocessor for One-Hot Encoding
trf = ColumnTransformer([
    ('trf', OneHotEncoder(sparse_output=False, drop='first'), ['batting_team', 'bowling_team', 'city'])
], remainder='passthrough')

# Logistic Regression Pipeline
pipe_lr = Pipeline(steps=[
    ('step1', trf),
    ('step2', LogisticRegression(solver='liblinear'))
])

pipe_lr.fit(X_train, y_train)
y_pred_lr = pipe_lr.predict(X_test)
y_prob_lr = pipe_lr.predict_proba(X_test)

print("--- Logistic Regression Baseline ---")
print(f"Accuracy: {accuracy_score(y_test, y_pred_lr):.4f}")
print(f"Log Loss: {log_loss(y_test, y_prob_lr):.4f}")

--- Logistic Regression Baseline ---
Accuracy: 0.8006
Log Loss: 0.4189


In [8]:
# XGBoost Classifier
xgb_model = xgb.XGBClassifier(n_estimators=100, learning_rate=0.1, max_depth=5, random_state=42)

# Wrap XGBoost in a Calibrator for better probability outputs
calibrated_xgb = CalibratedClassifierCV(xgb_model, method='sigmoid', cv=5)

# Upgraded Pipeline
pipe_xgb = Pipeline(steps=[
    ('step1', trf),
    ('step2', calibrated_xgb)
])

pipe_xgb.fit(X_train, y_train)
y_pred_xgb = pipe_xgb.predict(X_test)
y_prob_xgb = pipe_xgb.predict_proba(X_test)

print("--- Calibrated XGBoost ---")
print(f"Accuracy: {accuracy_score(y_test, y_pred_xgb):.4f}")
print(f"Log Loss: {log_loss(y_test, y_prob_xgb):.4f}") 
# Note: A lower Log Loss indicates better, more reliable probabilities!

--- Calibrated XGBoost ---
Accuracy: 0.9137
Log Loss: 0.1948


In [9]:
# Let's simulate a match scenario
def predict_match_probability(model, batting_team, bowling_team, city, target, current_score, balls_left, wickets_left):
    runs_left = target - current_score
    crr = (current_score * 6) / (120 - balls_left)
    rrr = (runs_left * 6) / balls_left
    
    # Create a DataFrame for the single prediction
    input_data = pd.DataFrame({
        'batting_team': [batting_team],
        'bowling_team': [bowling_team],
        'city': [city],
        'runs_left': [runs_left],
        'balls_left': [balls_left],
        'wickets_left': [wickets_left],
        'total_runs_x': [target],
        'crr': [crr],
        'rrr': [rrr]
    })
    
    # Get the probability of class 1 (Batting team winning)
    win_prob = model.predict_proba(input_data)[0][1]
    loss_prob = model.predict_proba(input_data)[0][0]
    
    print(f"{batting_team} Win Probability: {round(win_prob * 100, 1)}%")
    print(f"{bowling_team} Win Probability: {round(loss_prob * 100, 1)}%")

# Test the function with the Calibrated XGBoost model
predict_match_probability(pipe_xgb, 'Chennai Super Kings', 'Mumbai Indians', 'Mumbai', 180, 120, 30, 6)

Chennai Super Kings Win Probability: 21.4%
Mumbai Indians Win Probability: 78.6%


In [10]:
import pickle

# Save the calibrated XGBoost pipeline to a file named 'pipe.pkl'
with open('pipe.pkl', 'wb') as file:
    pickle.dump(pipe_xgb, file)

In [11]:
%%writefile app.py

import streamlit as st
import pickle
import pandas as pd

# --- 1. Load the Model ---
# We load the pickle file we just created in the Jupyter Notebook
pipe = pickle.load(open('pipe.pkl', 'rb'))

# --- 2. Define the UI Data ---
teams = [
    'Sunrisers Hyderabad', 'Mumbai Indians', 'Royal Challengers Bangalore',
    'Kolkata Knight Riders', 'Punjab Kings', 'Chennai Super Kings',
    'Rajasthan Royals', 'Delhi Capitals', 'Gujarat Titans', 'Lucknow Super Giants'
]

cities = ['Hyderabad', 'Bangalore', 'Mumbai', 'Indore', 'Kolkata', 'Delhi',
       'Chandigarh', 'Jaipur', 'Chennai', 'Cape Town', 'Port Elizabeth',
       'Durban', 'Centurion', 'East London', 'Johannesburg', 'Kimberley',
       'Bloemfontein', 'Ahmedabad', 'Cuttack', 'Nagpur', 'Dharamsala',
       'Visakhapatnam', 'Pune', 'Raipur', 'Ranchi', 'Abu Dhabi',
       'Sharjah', 'Mohali', 'Bengaluru']

# --- 3. Build the UI Layout ---
st.title('IPL Win Probability Predictor 🏏')

col1, col2 = st.columns(2)

with col1:
    batting_team = st.selectbox('Select the Batting Team', sorted(teams))
with col2:
    bowling_team = st.selectbox('Select the Bowling Team', sorted(teams))

selected_city = st.selectbox('Select Host City', sorted(cities))

target = st.number_input('Target Score (1st Innings Total + 1)', step=1, min_value=1)

col3, col4, col5 = st.columns(3)

with col3:
    score = st.number_input('Current Score', step=1, min_value=0)
with col4:
    overs = st.number_input('Overs Completed', step=0.1, min_value=0.0, max_value=20.0)
with col5:
    wickets = st.number_input('Wickets Down', step=1, min_value=0, max_value=10)

# --- 4. Prediction Logic ---
if st.button('Predict Probability'):
    # Prevent division by zero and invalid states
    if overs == 0:
        st.error("Please enter overs > 0 to calculate run rates.")
    elif score > target:
        st.success(f"{batting_team} has already won!")
    elif wickets == 10 and score < target:
        st.error(f"{bowling_team} has already won!")
    else:
        # Calculate dynamic features
        runs_left = target - score
        balls_left = 120 - int(overs * 6) # Rough estimation for simplicity in UI
        wickets_left = 10 - wickets
        crr = (score * 6) / (120 - balls_left)
        rrr = (runs_left * 6) / balls_left

        # Create input dataframe matching the exact columns the model expects
        input_df = pd.DataFrame({'batting_team': [batting_team], 'bowling_team': [bowling_team],
                                 'city': [selected_city], 'runs_left': [runs_left], 
                                 'balls_left': [balls_left], 'wickets_left': [wickets_left], 
                                 'total_runs_x': [target], 'crr': [crr], 'rrr': [rrr]})

        # Get probabilities
        result = pipe.predict_proba(input_df)
        loss = result[0][0]
        win = result[0][1]

        # Display results nicely
        st.header(f"{batting_team} Win Probability: {round(win * 100, 1)}%")
        st.header(f"{bowling_team} Win Probability: {round(loss * 100, 1)}%")

Writing app.py
